# 8.2 · Production checklist and system-design interviews

**生产清单与系统设计面试**

> **After this lesson you can:** Take an MCP system to production — stateless deployment, auth, rate limits, tracing, caching, versioning, testing and rollout — and answer senior MCP system-design questions in a structured way.

Official reading: [Security best practices](https://modelcontextprotocol.io/docs/2026-07-28/tutorials/security/security_best_practices) · [Spec: Streamable HTTP](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http) · [Roadmap](https://modelcontextprotocol.io/development/roadmap)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 0. Setup
We will run **seven** HTTP services on `127.0.0.1`, each in a daemon thread: two knowledge-base replicas, a round-robin load balancer, a CRM backend, the gateway, and (in section 6) two more replicas plus a second balancer. `serve()` starts one and waits until it accepts connections.

In [ ]:
import itertools
import json
import logging
import secrets
import socket
import statistics
import tempfile
import threading
import time
from collections import Counter
from contextlib import asynccontextmanager
from pathlib import Path

import anyio
import httpx
import httpx2
import mcp_types as types
import uvicorn
from pydantic import BaseModel
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import Response
from starlette.routing import Route

from mcp import Client
from mcp.client.streamable_http import streamable_http_client
from mcp.server import MCPServer
from mcp.server.auth.middleware.auth_context import get_access_token
from mcp.server.auth.provider import AccessToken
from mcp.server.auth.settings import AuthSettings
from mcp.server.lowlevel import Server
from mcp.server.mcpserver import Context, RequestStateSecurity
from mcp.server.mcpserver.exceptions import ToolError
from mcp.shared.exceptions import MCPError

WORKDIR = Path(tempfile.mkdtemp(prefix="gateway-"))
RUNNING: list[uvicorn.Server] = []


def quiet_logs():
    for name in ("httpx", "httpx2", "uvicorn", "mcp"):
        logging.getLogger(name).setLevel(logging.CRITICAL)


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def serve(app, port: int) -> str:
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.01)
    RUNNING.append(server)
    quiet_logs()
    return f"http://127.0.0.1:{port}/mcp"


def show(obj, limit=1200):
    if isinstance(obj, BaseModel):
        obj = obj.model_dump(by_alias=True, exclude_none=True, mode="json")
    text = json.dumps(obj, indent=2) if not isinstance(obj, str) else obj
    print(text if len(text) <= limit else text[:limit] + f"\n… [{len(text) - limit} more chars]")


quiet_logs()
print("ready; audit log will be written under", WORKDIR)

### 1. Backends that only trust the gateway
Two internal MCP servers sit behind the gateway: a knowledge base (`kb`, two replicas) and a CRM. Each is an OAuth **resource server** that accepts only a **service token issued for itself** (audience = its public URL). The gateway authenticates with its *own* credentials — it never forwards the user's token (token passthrough is forbidden). The user's identity travels only as an audit hint in a vendor-prefixed `_meta` key.

In [ ]:
KB_PORT, CRM_PORT, GW_PORT = free_port(), free_port(), free_port()
KB_URL = f"http://127.0.0.1:{KB_PORT}/mcp"          # public URL of the kb service = its load balancer
CRM_URL = f"http://127.0.0.1:{CRM_PORT}/mcp"
GW_URL = f"http://127.0.0.1:{GW_PORT}/mcp"

# service credentials minted for the gateway, one per backend audience (a stand-in for an OAuth AS)
SERVICE_TOKENS = {"svc-gw-kb-1f9c": KB_URL, "svc-gw-crm-77ab": CRM_URL}
BACKEND_SAW: list[dict] = []                       # what backends observed, for the tests


class ServiceTokenVerifier:
    def __init__(self, audience: str):
        self.audience = audience

    async def verify_token(self, token: str) -> AccessToken | None:
        aud = SERVICE_TOKENS.get(token)
        if aud is None:
            return None
        return AccessToken(token=token, client_id="mcp-gateway", scopes=["backend"], subject="mcp-gateway",
                           resource=aud)            # the SDK rejects it unless aud == resource_server_url


def backend_auth(url: str) -> dict:
    return {"token_verifier": ServiceTokenVerifier(url),
            "auth": AuthSettings(issuer_url="https://idp.example.com", resource_server_url=url,
                                 validate_token_resource=True)}


def observe(ctx: Context, replica: str):
    meta = ctx.request_context.meta or {}
    BACKEND_SAW.append({"replica": replica, "traceparent": meta.get("traceparent"),
                        "on_behalf_of": meta.get("com.example.gateway/user")})


class KbAnswer(BaseModel):
    replica: str
    hits: list[str]


class Customer(BaseModel):
    customer_id: str
    name: str
    tier: str


def make_kb(replica: str, request_state_keys: list[bytes] | None = None) -> MCPServer:
    extra = {"request_state_security": RequestStateSecurity(keys=request_state_keys)} if request_state_keys else {}
    kb = MCPServer("kb", log_level="WARNING", **backend_auth(KB_URL), **extra)

    @kb.tool()
    async def search_kb(query: str, ctx: Context) -> KbAnswer:
        """Search internal knowledge-base articles by keyword."""
        observe(ctx, replica)
        await anyio.sleep(0.01)                        # pretend to do some I/O
        return KbAnswer(replica=replica, hits=[f"KB-{abs(hash(query)) % 1000}: how to {query}"])

    @kb.tool()
    async def reindex(ctx: Context) -> types.CallToolResult | types.InputRequiredResult:
        """Rebuild the search index (asks an operator to confirm)."""
        if not ctx.input_responses:
            return types.InputRequiredResult(
                input_requests={"confirm": types.ElicitRequest(params=types.ElicitRequestFormParams(
                    message="Rebuild the whole index?",
                    requested_schema={"type": "object", "properties": {"ok": {"type": "boolean"}},
                                      "required": ["ok"]}))},
                request_state=json.dumps({"asked_by": replica}))
        asked_by = json.loads(ctx.request_state)["asked_by"]
        return types.CallToolResult(content=[types.TextContent(
            type="text", text=f"reindex confirmed: asked by replica {asked_by}, finished by replica {replica}")])

    return kb


CUSTOMERS = {"c-42": {"name": "Acme Corp", "tier": "gold"}, "c-7": {"name": "Globex", "tier": "silver"}}
crm = MCPServer("crm", log_level="WARNING", **backend_auth(CRM_URL))


@crm.tool()
def lookup_customer(customer_id: str, ctx: Context) -> Customer:
    """Look up a customer record by id."""
    observe(ctx, "crm")
    if customer_id not in CUSTOMERS:
        raise ToolError(f"no customer {customer_id}")
    return Customer(customer_id=customer_id, **CUSTOMERS[customer_id])


@crm.tool()
def update_tier(customer_id: str, tier: str, ctx: Context) -> Customer:
    """Change a customer's support tier. Writes to the system of record."""
    observe(ctx, "crm")
    CUSTOMERS[customer_id]["tier"] = tier
    return Customer(customer_id=customer_id, **CUSTOMERS[customer_id])


serve(crm.streamable_http_app(), CRM_PORT)
print("crm up at", CRM_URL)

### 2. Two replicas behind a round-robin balancer
A 2026-07-28 server keeps no per-connection state, so a dumb **round-robin** balancer is enough: no sticky sessions, no `Mcp-Session-Id` affinity. The balancer below forwards each POST to the next replica and copies the MCP headers. One client, consecutive calls, different replicas — and nothing breaks.

In [ ]:
def round_robin_balancer(upstreams: list[str]) -> Starlette:
    ring, http = itertools.cycle(upstreams), httpx.AsyncClient(trust_env=False)
    hop_by_hop = {"host", "content-length", "connection"}

    async def proxy(request: Request) -> Response:
        upstream = next(ring)
        headers = {k: v for k, v in request.headers.items() if k.lower() not in hop_by_hop}
        r = await http.request(request.method, upstream, content=await request.body(), headers=headers)
        keep = {k: v for k, v in r.headers.items() if k.lower() in ("content-type", "www-authenticate")}
        return Response(r.content, status_code=r.status_code, headers=keep)

    return Starlette(routes=[Route("/mcp", proxy, methods=["POST", "GET", "DELETE"])])


KB_REPLICAS = {name: serve(make_kb(name).streamable_http_app(), free_port()) for name in ("kb-a", "kb-b")}
serve(round_robin_balancer(list(KB_REPLICAS.values())), KB_PORT)


async def service_client_demo():
    async with httpx2.AsyncClient(headers={"Authorization": "Bearer svc-gw-kb-1f9c"}, trust_env=False) as http:
        async with Client(streamable_http_client(KB_URL, http_client=http)) as kb:
            for q in ("reset a password", "rotate an API key", "export invoices", "close an account"):
                r = await kb.call_tool("search_kb", {"query": q})
                print(f"{q:<20} -> answered by {r.structured_content['replica']}")


await service_client_demo()

with httpx.Client(trust_env=False) as h:     # audience check: the CRM token is not valid for kb
    probe = {"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": {
        "io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}}}}
    r = h.post(KB_URL, json=probe, headers={"Authorization": "Bearer svc-gw-crm-77ab", "MCP-Protocol-Version": "2026-07-28",
                                            "Mcp-Method": "server/discover", "Accept": "application/json, text/event-stream"})
    print("\ncrm token presented to kb ->", r.status_code, r.headers.get("www-authenticate", "")[:60], "…")

### 3. The gateway: one front door, policy in the middle
The gateway is itself an MCP server (built on the low-level `Server`, so it can pass backend schemas through unchanged) and an MCP client of each backend. On every request it:

1. **authenticates** the bearer token (SDK `token_verifier` → `401` + `WWW-Authenticate` otherwise);
2. lists only the tools the user's **role allows**, with `cacheScope: "private"` because the list differs per user;
3. re-checks the **allowlist** on `tools/call` and applies a per-user **token-bucket rate limit**;
4. continues the caller's **trace** (`traceparent` in `_meta`) or starts a new one, and sends a child span to the backend;
5. forwards with its **own service token**, never the user's;
6. writes one **structured audit record** (JSON line) per call.

In [ ]:
USERS = {   # gateway access tokens -> (user, role); a stand-in for the enterprise IdP
    "tok-ana": ("ana", "support"),
    "tok-ben": ("ben", "sales-admin"),
    "tok-load": ("loadbot", "support"),
}
ROLE_TOOLS = {
    "support": {"kb__search_kb", "crm__lookup_customer"},
    "sales-admin": {"kb__search_kb", "crm__lookup_customer", "crm__update_tier"},
}
RATE = {"default": (5, 5.0), "loadbot": (200, 500.0)}   # (burst capacity, refill tokens per second)
BACKENDS = {"kb": (KB_URL, "svc-gw-kb-1f9c"), "crm": (CRM_URL, "svc-gw-crm-77ab")}
AUDIT_FILE = WORKDIR / "audit.jsonl"
AUDIT: list[dict] = []


class UserTokenVerifier:
    async def verify_token(self, token: str) -> AccessToken | None:
        if token not in USERS:
            return None
        user, role = USERS[token]
        return AccessToken(token=token, client_id="agent-host", subject=user, scopes=["mcp"],
                           claims={"role": role}, resource=GW_URL)


class TokenBucket:
    def __init__(self, capacity: int, refill_per_s: float):
        self.capacity, self.refill, self.tokens, self.t = capacity, refill_per_s, float(capacity), time.monotonic()

    def take(self) -> float:
        """0.0 if allowed, else seconds until a token is available."""
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.refill)
        self.t = now
        if self.tokens >= 1:
            self.tokens -= 1
            return 0.0
        return (1 - self.tokens) / self.refill


BUCKETS: dict[str, TokenBucket] = {}
CATALOG = {"tools": None, "fresh_until": 0.0}


POOLS: dict[str, httpx2.AsyncClient] = {}       # one pooled HTTP client per backend (keep-alive connections)


def backend_client(name: str) -> Client:
    url, service_token = BACKENDS[name]
    if name not in POOLS:                          # created lazily, inside the gateway's event loop
        POOLS[name] = httpx2.AsyncClient(headers={"Authorization": f"Bearer {service_token}"}, trust_env=False,
                                         limits=httpx2.Limits(max_connections=50))
    # mode pins the version: no server/discover probe per call (the gateway already knows its backends).
    # A Client is cheap and stateless here: open one per call; the pooled HTTP client reuses connections.
    return Client(streamable_http_client(url, http_client=POOLS[name]), mode="2026-07-28")


async def backend_catalog() -> dict[str, types.Tool]:
    if CATALOG["tools"] is None or time.monotonic() > CATALOG["fresh_until"]:
        tools, ttl = {}, 3_600_000
        for name in BACKENDS:
            async with backend_client(name) as c:
                listing = await c.list_tools()
            ttl = min(ttl, listing.ttl_ms or 60_000)
            for t in listing.tools:
                tools[f"{name}__{t.name}"] = t.model_copy(update={"name": f"{name}__{t.name}"})
        CATALOG.update(tools=tools, fresh_until=time.monotonic() + ttl / 1000)
    return CATALOG["tools"]


def caller() -> tuple[str, str]:
    token = get_access_token()                      # set by the SDK's bearer-auth middleware
    return token.subject, token.claims["role"]


def child_trace(meta: dict) -> tuple[str, str | None, str]:
    """Continue an incoming W3C traceparent, or start a new trace. Returns (trace_id, parent_span, span)."""
    parts = str(meta.get("traceparent", "")).split("-")
    if len(parts) == 4 and len(parts[1]) == 32 and len(parts[2]) == 16:
        return parts[1], parts[2], secrets.token_hex(8)
    return secrets.token_hex(16), None, secrets.token_hex(8)


def audit(record: dict):
    AUDIT.append(record)
    with AUDIT_FILE.open("a") as f:
        f.write(json.dumps(record) + "\n")


async def gw_list_tools(ctx, params) -> types.ListToolsResult:
    user, role = caller()
    allowed = [t for name, t in (await backend_catalog()).items() if name in ROLE_TOOLS[role]]
    return types.ListToolsResult(tools=allowed, ttl_ms=60_000, cache_scope="private")


async def gw_call_tool(ctx, params) -> types.CallToolResult:
    user, role = caller()
    meta = dict(ctx.meta or {})
    trace_id, parent, span = child_trace(meta)
    record = {"ts": round(time.time(), 3), "user": user, "role": role, "tool": params.name,
              "trace_id": trace_id, "parent_span": parent, "span": span}
    started = time.perf_counter()

    def finish(outcome: str, result: types.CallToolResult) -> types.CallToolResult:
        record.update(outcome=outcome, ms=round((time.perf_counter() - started) * 1000, 1))
        audit(record)
        return result

    catalog = await backend_catalog()
    if params.name not in catalog:
        record.update(outcome="unknown_tool", ms=0.0)
        audit(record)
        raise MCPError(types.INVALID_PARAMS, f"Unknown tool: {params.name}")      # protocol error, per spec
    if params.name not in ROLE_TOOLS[role]:
        return finish("denied", types.CallToolResult(is_error=True, content=[types.TextContent(
            type="text", text=f"Role '{role}' may not call {params.name}.")]))
    wait = BUCKETS.setdefault(user, TokenBucket(*RATE.get(user, RATE["default"]))).take()
    if wait:
        return finish("rate_limited", types.CallToolResult(is_error=True, content=[types.TextContent(
            type="text", text=f"Rate limit exceeded; retry in {wait:.2f}s.")]))
    backend, _, tool = params.name.partition("__")
    # NOTE: the SDK client also injects the *ambient OpenTelemetry context* into outgoing _meta. When the caller
    # sent a traceparent, the SDK server middleware made it ambient, so that value wins over ours (section 4).
    forward_meta = {"traceparent": f"00-{trace_id}-{span}-01", "com.example.gateway/user": user}
    for key in ("tracestate", "baggage"):              # propagate the rest of the W3C context untouched
        if key in meta:
            forward_meta[key] = meta[key]
    async with backend_client(backend) as c:
        result = await c.call_tool(tool, params.arguments or {}, meta=forward_meta)
    record["backend"] = backend
    if isinstance(result.structured_content, dict) and "replica" in result.structured_content:
        record["replica"] = result.structured_content["replica"]
    return finish("tool_error" if result.is_error else "ok", types.CallToolResult(
        content=result.content, structured_content=result.structured_content, is_error=result.is_error))


gateway = Server("mcp-gateway", version="1.4.0", on_list_tools=gw_list_tools, on_call_tool=gw_call_tool)
serve(gateway.streamable_http_app(
    auth=AuthSettings(issuer_url="https://idp.example.com", resource_server_url=GW_URL,
                      validate_token_resource=True, required_scopes=["mcp"]),
    token_verifier=UserTokenVerifier()), GW_PORT)
print("gateway up at", GW_URL)

### 4. Use the gateway as two different users
`ana` (support) and `ben` (sales-admin) see different tool lists from the same endpoint. Ana's host sends its own `traceparent`, so the gateway *continues* her trace; Ben's doesn't, so the gateway *starts* one.

In [ ]:
@asynccontextmanager
async def user_client(token: str):
    """An agent host's connection to the gateway. The SDK does not close an http_client you pass in."""
    async with httpx2.AsyncClient(headers={"Authorization": f"Bearer {token}"}, trust_env=False) as http:
        async with Client(streamable_http_client(GW_URL, http_client=http)) as client:
            yield client


ANA_TRACE = "4bf92f3577b34da6a3ce929d0e0e4736"
async with user_client("tok-ana") as ana, user_client("tok-ben") as ben:
    ana_tools = await ana.list_tools()
    ben_tools = await ben.list_tools()
    print("ana sees:", [t.name for t in ana_tools.tools], "| cacheScope:", ana_tools.cache_scope)
    print("ben sees:", [t.name for t in ben_tools.tools])

    r = await ana.call_tool("crm__lookup_customer", {"customer_id": "c-42"},
                            meta={"traceparent": f"00-{ANA_TRACE}-00f067aa0ba902b7-01", "baggage": "tenant=acme"})
    print("\nana lookup ->", r.structured_content)
    denied = await ana.call_tool("crm__update_tier", {"customer_id": "c-42", "tier": "platinum"})
    print("ana update ->", denied.is_error, denied.content[0].text)
    ok = await ben.call_tool("crm__update_tier", {"customer_id": "c-42", "tier": "platinum"})
    print("ben update ->", ok.structured_content)
    try:
        await ana.call_tool("crm__drop_database", {})
    except MCPError as exc:
        print("unknown    ->", exc.code, exc.message)

print("\naudit.jsonl:")
for line in AUDIT_FILE.read_text().splitlines():
    print(" ", line)
print("\nwhat the CRM backend saw:")
for seen in BACKEND_SAW[-2:]:
    print(" ", seen)

Read the two `traceparent` values the CRM saw. Ben's request carries the gateway's **child span** — the value our gateway put in `_meta`. Ana's carries her host's **original span id**, not the gateway's. Why? The Python SDK has OpenTelemetry built in: its server middleware *extracts* `traceparent` from incoming `_meta` into the ambient context, and its client *injects* the ambient context into outgoing `_meta`, overwriting what you passed. With only the OTel API installed (as here) the tracer is a no-op, so the injected span is the caller's. In production you install the OTel SDK with a `TracerProvider`: then the SDK creates real `SERVER` and `CLIENT` spans and this propagation is exactly what you want. Either way the **trace id** is preserved end to end, and that is what joins the logs.

### 5. Rate limiting and authentication at the edge
Ana's bucket holds 5 tokens and refills at 5/s: a burst of 12 calls gets throttled with an actionable `isError` message (an agent can back off). A missing or foreign token never reaches the MCP layer: the SDK middleware answers `401` with the Protected Resource Metadata URL (lesson 6.1).

In [ ]:
async with user_client("tok-ana") as ana:
    results = []

    async def one(i):
        r = await ana.call_tool("kb__search_kb", {"query": f"burst {i}"})
        results.append("throttled" if r.is_error else "ok")

    async with anyio.create_task_group() as tg:
        for i in range(12):
            tg.start_soon(one, i)
print("burst of 12 ->", Counter(results))

with httpx.Client(trust_env=False) as h:
    for label, token in [("no token", None), ("backend service token", "svc-gw-kb-1f9c")]:
        headers = {"Accept": "application/json, text/event-stream", "MCP-Protocol-Version": "2026-07-28",
                   "Mcp-Method": "tools/list"}
        if token:
            headers["Authorization"] = f"Bearer {token}"
        r = h.post(GW_URL, headers=headers, json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {
            "_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
                      "io.modelcontextprotocol/clientCapabilities": {}}}})
        print(f"{label:<22} -> {r.status_code}  {r.headers.get('www-authenticate', '')[:95]}")

### 6. Load test: statelessness under concurrency
Two independent agent hosts fire 40 concurrent calls each through the gateway. Because no request depends on a previous one, the balancer spreads them across both kb replicas — and **each client is served by both replicas**. We also compute latency percentiles, the shape of the numbers you'd put on a dashboard. (All eight servers share one Python process here, so the absolute numbers say nothing about real capacity; the distribution and the zero error count are the point.)

In [ ]:
LOAD_RESULTS: list[dict] = []


async def agent_host(host_id: str, n_calls: int):
    async with user_client("tok-load") as c:
        async def one(i):
            t0 = time.perf_counter()
            r = await c.call_tool("kb__search_kb", {"query": f"{host_id} question {i}"})
            LOAD_RESULTS.append({"host": host_id, "replica": r.structured_content["replica"],
                                 "ms": (time.perf_counter() - t0) * 1000, "error": r.is_error})

        async with anyio.create_task_group() as tg:
            for i in range(n_calls):
                tg.start_soon(one, i)


started = time.perf_counter()
async with anyio.create_task_group() as tg:
    tg.start_soon(agent_host, "host-1", 40)
    tg.start_soon(agent_host, "host-2", 40)
elapsed = time.perf_counter() - started

latencies = sorted(r["ms"] for r in LOAD_RESULTS)
print(f"{len(LOAD_RESULTS)} calls in {elapsed:.2f}s  ->  {len(LOAD_RESULTS) / elapsed:.0f} calls/s, "
      f"errors: {sum(r['error'] for r in LOAD_RESULTS)}")
print(f"latency p50 {statistics.median(latencies):.0f} ms | p95 {latencies[int(0.95 * len(latencies)) - 1]:.0f} ms")
for host_id in ("host-1", "host-2"):
    print(host_id, "served by", dict(Counter(r["replica"] for r in LOAD_RESULTS if r["host"] == host_id)))

### 7. The one piece of shared state: MRTR keys across replicas
Statelessness has a catch. `reindex` returns an `InputRequiredResult`; the retry carries a `requestState` that the SDK **seals with a key**. By default each process generates its own key (`RequestStateSecurity.ephemeral()`), so when the round-robin balancer sends the retry to the *other* replica, it cannot open the state. The fix is configuration, not stickiness: give every replica the same key ring.

In [ ]:
async def operator_says_yes(ctx, params):
    return types.ElicitResult(action="accept", content={"ok": True})


async def try_reindex(url: str) -> str:
    async with httpx2.AsyncClient(headers={"Authorization": "Bearer svc-gw-kb-1f9c"}, trust_env=False) as http:
        async with Client(streamable_http_client(url, http_client=http),
                          elicitation_callback=operator_says_yes, mode="2026-07-28") as kb:
            try:
                r = await kb.call_tool("reindex", {})
                return r.content[0].text
            except MCPError as exc:
                return f"MCPError {exc.code}: {exc.message}"


print("per-process keys :", await try_reindex(KB_URL))

SHARED_KEYS = [secrets.token_bytes(32)]           # in production: from a secret manager, rotated as a ring
shared = [serve(make_kb(n, request_state_keys=SHARED_KEYS).streamable_http_app(), free_port())
          for n in ("kb-c", "kb-d")]
KB2_PORT = free_port()
serve(round_robin_balancer(shared), KB2_PORT)
SERVICE_TOKENS["svc-gw-kb-1f9c"] = KB_URL          # (same audience: the replicas still verify against KB_URL)
print("shared key ring  :", await try_reindex(f"http://127.0.0.1:{KB2_PORT}/mcp"))

### 8. Pre-rollout smoke tests (what CI runs before traffic shifts)
A production checklist is only real if it is executable. These checks hit the gateway with raw HTTP and assert the transport-level guarantees from the spec, then the policy guarantees from our design.

In [ ]:
BASE_HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
                "MCP-Protocol-Version": "2026-07-28", "Authorization": "Bearer tok-ben"}


def rpc(method, params, headers=None, http_method="POST"):
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": {
        **params, "_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
                            "io.modelcontextprotocol/clientCapabilities": {}}}}
    h = {**BASE_HEADERS, "Mcp-Method": method, **(headers or {})}
    with httpx.Client(trust_env=False) as client:
        return client.request(http_method, GW_URL, json=body if http_method == "POST" else None, headers=h)


call = {"name": "kb__search_kb", "arguments": {"query": "x"}}
checks = [
    ("valid tools/call -> 200", lambda: rpc("tools/call", call, {"Mcp-Name": "kb__search_kb"}).status_code == 200),
    ("missing token -> 401", lambda: rpc("tools/call", call, {"Mcp-Name": "kb__search_kb", "Authorization": ""}).status_code == 401),
    ("Mcp-Name mismatch -> 400 / -32020",
     lambda: rpc("tools/call", call, {"Mcp-Name": "crm__update_tier"}).json()["error"]["code"] == -32020),
    ("foreign Origin -> 403 (DNS rebinding)",
     lambda: rpc("tools/call", call, {"Mcp-Name": "kb__search_kb", "Origin": "https://evil.example"}).status_code == 403),
    ("GET stream removed -> 405", lambda: rpc("tools/list", {}, http_method="GET").status_code == 405),
    ("unknown method -> 404 / -32601", lambda: rpc("tasks/frobnicate", {}).status_code == 404),
    ("tools/list is private-scoped", lambda: rpc("tools/list", {}).json()["result"]["cacheScope"] == "private"),
    ("serverInfo stamped in _meta",
     lambda: rpc("tools/list", {}).json()["result"]["_meta"]["io.modelcontextprotocol/serverInfo"]["version"] == "1.4.0"),
]
for label, check in checks:
    print(("PASS " if check() else "FAIL ") + label)

### 9. End-to-end assertions on the audit trail and traces

In [ ]:
def test_trace_continued_for_ana():
    rec = next(r for r in AUDIT if r["user"] == "ana" and r["tool"] == "crm__lookup_customer")
    assert rec["trace_id"] == ANA_TRACE and rec["parent_span"] == "00f067aa0ba902b7"
    seen = [b for b in BACKEND_SAW if b["on_behalf_of"] == "ana" and b["replica"] == "crm"]
    assert seen[0]["traceparent"].split("-")[1] == ANA_TRACE      # one trace id from host -> gateway -> backend


def test_trace_started_for_ben():
    rec = next(r for r in AUDIT if r["user"] == "ben" and r["outcome"] == "ok")
    assert rec["parent_span"] is None and len(rec["trace_id"]) == 32


def test_policy_outcomes():
    outcomes = Counter(r["outcome"] for r in AUDIT)
    assert outcomes["denied"] == 1 and outcomes["unknown_tool"] == 1 and outcomes["rate_limited"] >= 1
    assert CUSTOMERS["c-42"]["tier"] == "platinum"                  # only ben's write went through


def test_backends_never_saw_user_tokens():
    assert all(b["on_behalf_of"] in {"ana", "ben", "loadbot", None} for b in BACKEND_SAW)
    assert "tok-" not in AUDIT_FILE.read_text()                      # no credentials in the audit log


def test_statelessness():
    for host_id in ("host-1", "host-2"):
        replicas = {r["replica"] for r in LOAD_RESULTS if r["host"] == host_id}
        assert replicas == {"kb-a", "kb-b"}, replicas
    assert not any(r["error"] for r in LOAD_RESULTS)


for test in (test_trace_continued_for_ana, test_trace_started_for_ben, test_policy_outcomes,
             test_backends_never_saw_user_tokens, test_statelessness):
    test()
    print("PASS", test.__name__)

In [ ]:
for server in RUNNING:
    server.should_exit = True
time.sleep(0.3)
print(f"stopped {len(RUNNING)} servers; {len(AUDIT)} audit records in {AUDIT_FILE}")

### Your turn
1. **Gateway-side caching.** `backend_catalog()` ignores `cacheScope`. Make it honour it: cache `public` backend listings once for everyone, but key `private` ones by user. What breaks if you share a `private` list across users?
2. **HTTP-native rate limiting.** Move the token bucket into a Starlette middleware that answers `429 Too Many Requests` with `Retry-After` *before* the MCP layer runs. What does the agent host gain, and what do you lose (hint: the model no longer sees a readable tool error)?
3. **Distributed limits.** Run two gateway replicas behind `round_robin_balancer`. Why does each user now get twice the burst? Sketch the fix (shared store such as Redis, or limits at the balancer).
4. **Version skew.** Point the balancer at one replica running a `kb` that renamed `search_kb` to `search_articles`. What does a cached tool list do, how does `ttlMs` bound the damage, and how would you roll out the rename safely (add, deprecate, remove)?